In [4]:
import numpy as np
import pandas as pd

class InformationGainFeatureSelector:

    
    def __init__(self, k, target_col):
        self.k = k
        self.target_col = target_col
        self.info_gains_ = {}
        self.selected_features_ = []
        
    # Compute entropy of a target variable
    def _entropy(self, y):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return -np.sum([p * np.log2(p) for p in probabilities if p > 0])
        
    def _conditional_entropy(self, X_col, y):
        unique_values, counts = np.unique(X_col, return_counts=True)
        conditional_ent = 0
        
        for value, count in zip(unique_values, counts):
            y_subset = y[X_col == value]
            weight = count / len(y)
            conditional_ent += weight * self._entropy(y_subset)
            
        return conditional_ent
        
    # Compute Information Gain for all features
    def _calculate_information_gain(self, df):
        y = df[self.target_col].values
        Hy = self._entropy(y)
        
        info_gains = {}
        
        for col in df.columns:
            if col == self.target_col:
                continue
                
            X_col = df[col].values
            info_gains[col] = Hy - self._conditional_entropy(X_col, y)
            
        return info_gains
        
    # Fit: calculate IG and rank features
    def fit(self, df):
        if self.target_col is None:
            raise ValueError("target_col must be provided.")
            
        self.info_gains_ = self._calculate_information_gain(df)
        
        # Sort features by IG in descending order
        sorted_features = sorted(
            self.info_gains_,
            key=self.info_gains_.get,
            reverse=True
        )
        
        self.selected_features_ = sorted_features[:self.k]
        
        return self
        
    def transform(self, df):
        return df[self.selected_features_ + [self.target_col]]
        
    def fit_transform(self, df):
        return self.fit(df).transform(df)


In [5]:
from sklearn.datasets import load_wine
import pandas as pd

# Load wine dataset
wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine['target'] = wine.target

# Instantiate InformationGainFeatureSelector
# We will select the top 5 features for demonstration
ig_selector = InformationGainFeatureSelector(k=5, target_col="target")

# Fit the selector
ig_selector.fit(df_wine)

# Get the transformed dataset with only selected features and target
df_transformed = ig_selector.transform(df_wine)
df_transformed.head()


,flavanoids,color_intensity,od280/od315_of_diluted_wines,alcohol,proline,target
0,3.06,5.64,3.92,14.23,1065.0,0
1,2.76,4.38,3.40,13.20,1050.0,0
2,3.24,5.68,3.17,13.16,1185.0,0
3,3.49,7.80,3.45,14.37,1480.0,0
4,2.69,4.32,2.93,13.24,735.0,0


In [6]:
# Display IG scores sorted from highest to lowest
pd.Series(ig_selector.info_gains_).sort_values(ascending=False)


flavanoids                      1.420755
color_intensity                 1.370083
od280/od315_of_diluted_wines    1.347050
alcohol                         1.340616
proline                         1.301545
malic_acid                      1.284201
total_phenols                   1.181265
proanthocyanins                 1.116149
hue                             1.041397
ash                             0.786340
alcalinity_of_ash               0.735794
magnesium                       0.636151
nonflavanoid_phenols            0.473907
dtype: float64